# Triển khai trên Google Colab – Version 0 (CPU tuần tự)

V0 chỉ chạy trên CPU nên build bằng `g++`, không cần GPU. Chọn `Runtime → Run all`.

- Mã nguồn: clone từ repository [`Panorama-Image-Stitching`](https://github.com/ZelusIsMe-05/Panorama-Image-Stitching) trên GitHub (repo public).
- Dataset: tải thẳng từ thư mục Google Drive bằng `gdown` (thư mục phải chia sẻ *Anyone with the link*), lưu vào `/content/dataset`.
- `PAIR_DIR`: đường dẫn tới **thư mục** của một cặp ảnh. Chương trình tự lấy 2 file `*.png` trong thư mục đó, sắp theo tên: file đứng trước là ảnh trái (A), file đứng sau là ảnh phải (B).

In [ ]:
REPO_URL = "https://github.com/ZelusIsMe-05/Panorama-Image-Stitching.git"
DATASET_URL = "https://drive.google.com/drive/folders/1eWi9OBGeIYzuW7wOhOMybv0-ZHNS6YrO"
RUNS = 5                                                              # số lần đo (+1 lần warm-up)

## 1. Lấy mã nguồn và dữ liệu

In [ ]:
import glob, os, re, subprocess
from IPython.display import Image, display
%cd -q /content

print("Đang clone mã nguồn từ GitHub...")
!rm -rf project && git clone -q --depth 1 {REPO_URL} project
print("  -> Xong: /content/project")

if os.path.isdir("dataset"):
    print("Dataset đã có trong /content/dataset, bỏ qua bước tải.")
else:
    print("Đang tải dataset từ Google Drive...")
    !gdown -q --folder {DATASET_URL} -O dataset --remaining-ok

PAIR_DIRS = sorted(d for d in {os.path.dirname(p) for p in glob.glob("/content/dataset/**/*.png", recursive=True)}
                   if len(glob.glob(os.path.join(d, "*.png"))) == 2)
print(f"  -> Xong: {len(PAIR_DIRS)} cặp ảnh trong /content/dataset")

## 2. Build chương trình

In [ ]:
!g++ -O3 -std=c++17 -x c++ project/src/main.cu -x none project/src/*.cpp -o stitch && echo "BUILD OK: /content/stitch"

## 3. Ghép cặp chỉ định

`pair_images` lấy 2 file `*.png` trong `PAIR_DIR` (báo lỗi nếu thư mục không có đúng 2 ảnh), rồi gọi
`./stitch <trái> <phải> <output> --version 0 --runs RUNS`. Cell hiển thị ảnh trái và ảnh phải, ảnh panorama **không crop** (chạy thêm 1 lần với `--no-crop`, giữ vùng đen để thấy hình dạng hai ảnh sau khi warp), ảnh panorama **có crop**, rồi bảng thời gian từng stage (đo trên lần chạy có crop).

In [ ]:
import base64, io
from IPython.display import HTML
from PIL import Image as PILImage

PAIR_DIR = "/content/dataset/medium/standard pairs/Outdoor/Pair 1"   # thư mục chứa 2 ảnh .png cần ghép

def pair_images(pair_dir):
    imgs = sorted(glob.glob(os.path.join(pair_dir, "*.png")))
    assert len(imgs) == 2, f"'{pair_dir}' cần có đúng 2 ảnh .png (đang có {len(imgs)})"
    return imgs

def stitch(pair_dir, out, runs=RUNS, crop=True):
    left, right = pair_images(pair_dir)
    cmd = ["./stitch", left, right, out, "--version", "0", "--runs", str(runs)]
    return subprocess.run(cmd if crop else cmd + ["--no-crop"], capture_output=True, text=True)

def figure(path, max_w=1000, caption=None):
    # Thu nhỏ thành JPEG trong bộ nhớ để notebook không bị nặng với ảnh 4K
    img = PILImage.open(path).convert("RGB")
    size = f"{img.width}×{img.height}"
    img.thumbnail((max_w, max_w))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=90)
    src = base64.b64encode(buf.getvalue()).decode()
    return (f'<figure style="flex:1; margin:0; text-align:center">'
            f'<img src="data:image/jpeg;base64,{src}" style="width:100%; height:auto">'
            f'<figcaption>{caption or os.path.basename(path)} ({size})</figcaption></figure>')

left, right = pair_images(PAIR_DIR)
print("Ảnh trái và ảnh phải:")
display(HTML(f'<div style="display:flex; gap:32px; max-width:1400px">{figure(left)}{figure(right)}</div>'))

p = stitch(PAIR_DIR, "/content/panorama.jpg")                                # đo thời gian (có crop)
q = stitch(PAIR_DIR, "/content/panorama_nocrop.jpg", runs=1, crop=False)     # chỉ để xem toàn bộ canvas
if p.returncode == 0 and q.returncode == 0:
    print("\nẢnh panorama (không crop, giữ vùng đen):")
    display(HTML(f'<div style="max-width:1400px">{figure("/content/panorama_nocrop.jpg", 1600, "Không crop")}</div>'))
    print("\nẢnh panorama (crop về hình chữ nhật lớn nhất không có pixel đen):")
    display(HTML(f'<div style="max-width:1400px">{figure("/content/panorama.jpg", 1600, "Có crop")}</div>'))
    lines = p.stdout.splitlines()
    print("\n".join(lines[next(i for i, l in enumerate(lines) if l.startswith("Image size")):]))
else:
    print(p.stdout, p.stderr, q.stdout, q.stderr)

## 4. Chạy toàn bộ dataset

Tổng thời gian pipeline (trung bình mỗi cặp, không tính đọc/ghi ảnh) của tất cả các cặp.

In [ ]:
total_ms, failed = 0.0, []
for d in PAIR_DIRS:
    p = stitch(d, "/content/tmp.jpg")
    m = re.search(r"Total \(end-to-end\)\s+([\d.]+)", p.stdout)
    if p.returncode == 0 and m:
        total_ms += float(m.group(1))
    else:
        failed.append(d)
print(f"{len(PAIR_DIRS) - len(failed)}/{len(PAIR_DIRS)} cặp thành công, tổng thời gian: {total_ms:.1f} ms ({total_ms / 1000:.2f} s)")
if failed:
    print("Lỗi:", *failed, sep="\n  ")